In [1]:
!pip install -q "transformers==4.44.2" "tokenizers<0.20" sentencepiece protobuf x-transformers optuna scikit-learn
# หลังรันเซลล์นี้ครั้งแรก: ให้ Restart & Clear (Session -> Restart) แล้วรันใหม่ตั้งแต่ต้น
# เพื่อให้ transformers เวอร์ชันที่ปักไว้ถูกโหลด (แก้บั๊ก tokenizer ของ transformers รุ่นใหม่)

# WangchanBERTa + x-transformers สำหรับ Prachatai multi-label 12 คลาส

Notebook นี้ดัดแปลงเฉพาะส่วน Dataset จาก SIED-Thai ให้ใช้ไฟล์ Prachatai ที่แบ่ง `train`, `validation` และ `test` มาแล้ว เพื่อเก็บผลการทดลองเพิ่มโดยไม่เปลี่ยนสถาปัตยกรรมหรือขั้นตอนฝึกของโมเดลต้นฉบับ

- Text column: `body_text`
- Labels: 12 หมวดข่าว Prachatai
- Loss: `BCEWithLogitsLoss`
- Prediction: `sigmoid + tuned threshold`
- Encoder: WangchanBERTa frozen
- Decoder/classifier: x-transformers ที่ฝึกและ optimize ด้วย Optuna
- Dataset split: ใช้ official Train/Validation/Test โดยตรง ไม่ split ใหม่ ไม่ลบข้อมูลซ้ำ และไม่ตัดแถว all-zero
- Training: 100 epochs ทุก Optuna trial และ 100 epochs สำหรับ Final model ไม่มี early stopping


In [2]:
import os

# กันดาวน์โหลดโมเดลจาก HuggingFace ค้าง/หลุด (ปิด hf_transfer + ตั้ง timeout)
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "60"

import random
import shutil
import subprocess
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from tqdm import tqdm

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModel,
    get_linear_schedule_with_warmup
)

from x_transformers import Decoder

from sklearn.metrics import (
    f1_score,
    accuracy_score,
    precision_score,
    recall_score,
    classification_report,
    hamming_loss
)

import optuna

In [3]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("DEVICE:", device)

DEVICE: cuda


In [5]:
import glob

MODEL_NAME = "airesearch/wangchanberta-base-att-spm-uncased"

MAX_LEN = 128

EPOCHS = 100

BATCH_SIZE = 64

NUM_LABELS = 12

N_TRIALS = 10

LABEL_COLUMNS = [
    "politics",
    "human_rights",
    "quality_of_life",
    "international",
    "social",
    "environment",
    "economics",
    "culture",
    "labor",
    "national_security",
    "ict",
    "education"
]

TEXT_COLUMN = "body_text"

# ===== Kaggle: ค้นหาไฟล์ CSV ใน /kaggle/input อัตโนมัติ (ไม่ต้องแก้ path เอง) =====
def _find_csv(filename):
    hits = glob.glob(f"/kaggle/input/**/{filename}", recursive=True)
    if not hits:
        raise FileNotFoundError(
            f"ไม่พบ {filename} ใน /kaggle/input — "
            f"ตรวจว่าได้ Add dataset เข้ามาใน notebook แล้วหรือยัง"
        )
    return hits[0]

TRAIN_CSV_PATH      = _find_csv("prachatai_train.csv")
VALIDATION_CSV_PATH = _find_csv("prachatai_validation.csv")
TEST_CSV_PATH       = _find_csv("prachatai_test.csv")

print("TRAIN CSV     :", TRAIN_CSV_PATH)
print("VALIDATION CSV:", VALIDATION_CSV_PATH)
print("TEST CSV      :", TEST_CSV_PATH)

TRAIN CSV     : /kaggle/input/datasets/kmkimmy/prachatai-dataset/prachatai_train.csv
VALIDATION CSV: /kaggle/input/datasets/kmkimmy/prachatai-dataset/prachatai_validation.csv
TEST CSV      : /kaggle/input/datasets/kmkimmy/prachatai-dataset/prachatai_test.csv


In [6]:
required_columns = [TEXT_COLUMN] + LABEL_COLUMNS


def load_prachatai_split(path, split_name):
    df = pd.read_csv(
        path,
        usecols=required_columns,
        low_memory=False
    )

    for col in required_columns:
        if col not in df.columns:
            raise ValueError(f"{split_name} ไม่มี column: {col}")

    df = df[required_columns].copy()
    df[TEXT_COLUMN] = df[TEXT_COLUMN].astype(str).str.strip()

    for col in LABEL_COLUMNS:
        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        ).fillna(0).astype(int)

        invalid_values = ~df[col].isin([0, 1])

        if invalid_values.any():
            values = df.loc[invalid_values, col].unique().tolist()
            raise ValueError(
                f"{split_name}/{col} มีค่านอกเหนือจาก 0/1: {values}"
            )

    return df.reset_index(drop=True)


train_df = load_prachatai_split(
    TRAIN_CSV_PATH,
    "train"
)

val_df = load_prachatai_split(
    VALIDATION_CSV_PATH,
    "validation"
)

test_df = load_prachatai_split(
    TEST_CSV_PATH,
    "test"
)

print("TRAIN     :", len(train_df))
print("VALIDATION:", len(val_df))
print("TEST      :", len(test_df))


TRAIN     : 54379
VALIDATION: 6721
TEST      : 6789


In [7]:
# ใช้ official split จาก Dataset โดยตรง ไม่มีการ split หรือย้ายข้อมูลข้ามชุด

print("TRAIN LABEL DISTRIBUTION")
print(train_df[LABEL_COLUMNS].sum())

print("\nVALIDATION LABEL DISTRIBUTION")
print(val_df[LABEL_COLUMNS].sum())

print("\nTEST LABEL DISTRIBUTION")
print(test_df[LABEL_COLUMNS].sum())

print("\nLABEL CARDINALITY")

print("TRAIN")
print(
    train_df[LABEL_COLUMNS]
    .sum(axis=1)
    .value_counts()
    .sort_index()
)

print("\nVALIDATION")
print(
    val_df[LABEL_COLUMNS]
    .sum(axis=1)
    .value_counts()
    .sort_index()
)

print("\nTEST")
print(
    test_df[LABEL_COLUMNS]
    .sum(axis=1)
    .value_counts()
    .sort_index()
)


TRAIN LABEL DISTRIBUTION
politics             31401
human_rights         12061
quality_of_life       9037
international         6432
social                6321
environment           6157
economics             3994
culture               3279
labor                 2905
national_security     2865
ict                   2326
education             2093
dtype: int64

VALIDATION LABEL DISTRIBUTION
politics             3852
human_rights         1458
quality_of_life      1144
international         828
social                782
environment           764
economics             487
culture               388
labor                 375
national_security     339
ict                   285
education             248
dtype: int64

TEST LABEL DISTRIBUTION
politics             3842
human_rights         1511
quality_of_life      1127
international         834
social                789
environment           772
economics             519
culture               398
labor                 350
national_security     3

In [8]:
# ===== โหลด WangchanBERTa แบบมี retry + ข้าม safetensors (กันดาวน์โหลดค้าง) =====
# repo มีทั้ง pytorch_model.bin และ model.safetensors (เนื้อหาเดียวกัน)
# ตัว safetensors มักค้าง จึงข้ามไป ใช้ .bin แทน แล้วชี้ MODEL_NAME ไป path ในเครื่อง
from huggingface_hub import snapshot_download

for _i in range(10):
    try:
        MODEL_NAME = snapshot_download(
            "airesearch/wangchanberta-base-att-spm-uncased",
            ignore_patterns=["*.safetensors"],
        )
        print("MODEL PATH:", MODEL_NAME)
        break
    except Exception as _e:
        print(f"retry {_i+1}: {_e}")
else:
    raise RuntimeError("โหลดโมเดลไม่สำเร็จ")

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

MODEL PATH: /root/.cache/huggingface/hub/models--airesearch--wangchanberta-base-att-spm-uncased/snapshots/b81d38df6b4755dbedec0bfea863c9956cbb963e


In [9]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [10]:
# ===== Precompute embeddings จาก frozen encoder ครั้งเดียว =====
_encoder = AutoModel.from_pretrained(MODEL_NAME).to(device).eval()
for p in _encoder.parameters():
    p.requires_grad = False

HIDDEN_SIZE = _encoder.config.hidden_size   # WangchanBERTa base = 768


class RawDataset(Dataset):
    """text -> input_ids/attention_mask (ใช้แค่ตอน precompute)"""
    def __init__(self, df):
        self.texts = df[TEXT_COLUMN].tolist()
        self.labels = df[LABEL_COLUMNS].values.astype(np.float32)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            str(self.texts[idx]),
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
            return_tensors="pt",
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.float),
        }


@torch.no_grad()
def precompute_embeddings(df, name):
    loader = DataLoader(RawDataset(df), batch_size=BATCH_SIZE, shuffle=False)
    embs, masks, labels = [], [], []
    use_amp = device.type == "cuda"
    for batch in tqdm(loader, desc=f"Encoding {name}"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        with torch.cuda.amp.autocast(enabled=use_amp):
            out = _encoder(input_ids=input_ids, attention_mask=attention_mask)
        embs.append(out.last_hidden_state.half().cpu())
        masks.append(attention_mask.cpu())
        labels.append(batch["labels"])
    return torch.cat(embs), torch.cat(masks), torch.cat(labels)


train_emb, train_mask, train_labels = precompute_embeddings(train_df, "train")
val_emb,   val_mask,   val_labels   = precompute_embeddings(val_df,   "val")
test_emb,  test_mask,  test_labels  = precompute_embeddings(test_df,  "test")

del _encoder
torch.cuda.empty_cache()

print("EMB SHAPE:", tuple(train_emb.shape), "| dtype:", train_emb.dtype)


Encoding test: 100%|██████████| 107/107 [01:09<00:00,  1.53it/s]


EMB SHAPE: (54379, 128, 768) | dtype: torch.float16


In [11]:
class CachedDataset(Dataset):
    """คืน embedding ที่ cache ไว้แล้ว (ไม่ต้องรัน encoder อีก)"""

    def __init__(self, emb, mask, labels):
        self.emb = emb
        self.mask = mask
        self.labels = labels

    def __len__(self):
        return len(self.emb)

    def __getitem__(self, idx):
        return {
            "embeddings": self.emb[idx],
            "attention_mask": self.mask[idx],
            "labels": self.labels[idx],
        }


train_dataset = CachedDataset(train_emb, train_mask, train_labels)

val_dataset = CachedDataset(val_emb, val_mask, val_labels)

test_dataset = CachedDataset(test_emb, test_mask, test_labels)

In [12]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [13]:
# pos_weight สำหรับ BCEWithLogitsLoss
# สูตรที่ใช้: negative / positive
# เหมาะกับ multi-label มากกว่า total / positive

label_counts = train_df[LABEL_COLUMNS].sum().values.astype(np.float32)

total_samples = len(train_df)

neg_counts = total_samples - label_counts

pos_weights = neg_counts / (label_counts + 1e-6)

pos_weights = torch.tensor(
    pos_weights,
    dtype=torch.float
).to(device)

print("LABEL COUNTS:", label_counts)
print("POS WEIGHTS :", pos_weights)


LABEL COUNTS: [31401. 12061.  9037.  6432.  6321.  6157.  3994.  3279.  2905.  2865.
  2326.  2093.]
POS WEIGHTS : tensor([ 0.7318,  3.5087,  5.0174,  7.4544,  7.6029,  7.8321, 12.6152, 15.5840,
        17.7191, 17.9805, 22.3788, 24.9814], device='cuda:0')


In [14]:
class AttentionPooling(nn.Module):

    def __init__(self, hidden_size):

        super().__init__()

        self.attention = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.Tanh(),
            nn.Linear(hidden_size, 1)
        )

    def forward(self, x, mask):

        scores = self.attention(x).squeeze(-1)

        # ใช้ค่าลบสุดของ dtype แทน -1e9 เพื่อไม่ให้ overflow ตอนรันเป็น fp16 (AMP)
        scores = scores.masked_fill(mask == 0, torch.finfo(scores.dtype).min)

        weights = torch.softmax(scores, dim=1)

        pooled = torch.sum(
            x * weights.unsqueeze(-1),
            dim=1
        )

        return pooled

In [15]:
class EmotionModel(nn.Module):

    def __init__(
        self,
        depth=1,
        heads=2,
        attn_dropout=0.2,
        ff_dropout=0.2,
        ff_mult=2
    ):

        super().__init__()

        # ไม่มี encoder แล้ว รับ embedding ที่ cache ไว้เข้ามาตรงๆ
        # (encoder WangchanBERTa ถูก freeze + precompute ไปแล้ว)

        self.decoder = Decoder(
            dim=HIDDEN_SIZE,
            depth=depth,
            heads=heads,
            attn_dropout=attn_dropout,
            ff_dropout=ff_dropout,
            ff_mult=ff_mult
        )

        self.pooling = AttentionPooling(HIDDEN_SIZE)

        self.dropout = nn.Dropout(0.4)

        self.fc = nn.Linear(
            HIDDEN_SIZE,
            NUM_LABELS
        )

    def forward(self, embeddings, attention_mask):

        # embeddings เป็น fp16 (มาจาก cache) -> cast เป็น float ให้ตรง dtype
        x = self.decoder(embeddings.float())

        x = self.pooling(x, attention_mask)

        x = self.dropout(x)

        logits = self.fc(x)

        return logits

In [16]:
def evaluate(model, loader, thresholds=None):

    model.eval()

    all_labels = []
    all_probs = []

    with torch.no_grad():

        for batch in loader:

            embeddings = batch["embeddings"].to(device)

            attention_mask = batch["attention_mask"].to(device)

            labels = batch["labels"].to(device)

            logits = model(
                embeddings,
                attention_mask
            )

            probs = torch.sigmoid(logits)

            all_probs.append(
                probs.cpu().numpy()
            )

            all_labels.append(
                labels.cpu().numpy()
            )

    all_probs = np.vstack(all_probs)

    all_labels = np.vstack(all_labels)

    if thresholds is None:

        thresholds = np.array(
            [0.5] * NUM_LABELS
        )

    preds = (
        all_probs >= thresholds
    ).astype(int)

    micro_f1 = f1_score(
        all_labels,
        preds,
        average="micro",
        zero_division=0
    )

    macro_f1 = f1_score(
        all_labels,
        preds,
        average="macro",
        zero_division=0
    )

    weighted_f1 = f1_score(
        all_labels,
        preds,
        average="weighted",
        zero_division=0
    )

    micro_precision = precision_score(
        all_labels,
        preds,
        average="micro",
        zero_division=0
    )

    macro_precision = precision_score(
        all_labels,
        preds,
        average="macro",
        zero_division=0
    )

    micro_recall = recall_score(
        all_labels,
        preds,
        average="micro",
        zero_division=0
    )

    macro_recall = recall_score(
        all_labels,
        preds,
        average="macro",
        zero_division=0
    )

    # สำหรับ multi-label accuracy_score คือ subset accuracy / exact match
    subset_accuracy = accuracy_score(
        all_labels,
        preds
    )

    h_loss = hamming_loss(
        all_labels,
        preds
    )

    return {
        "micro_f1": micro_f1,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "micro_precision": micro_precision,
        "macro_precision": macro_precision,
        "micro_recall": micro_recall,
        "macro_recall": macro_recall,
        "accuracy": subset_accuracy,
        "subset_accuracy": subset_accuracy,
        "hamming_loss": h_loss,
        "labels": all_labels,
        "preds": preds,
        "probs": all_probs
    }

In [17]:
def train_model(
    model,
    train_loader,
    val_loader,
    lr,
    weight_decay
):

    criterion = nn.BCEWithLogitsLoss(
        pos_weight=pos_weights
    )

    # encoder ถูก precompute แล้ว จึงเทรนเฉพาะ decoder + pooling + classifier
    trainable_params = [p for p in model.parameters() if p.requires_grad]

    optimizer = torch.optim.AdamW(
        trainable_params,
        lr=lr,
        weight_decay=weight_decay
    )

    total_steps = len(train_loader) * EPOCHS

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(
            0.1 * total_steps
        ),
        num_training_steps=total_steps
    )

    use_amp = device.type == "cuda"

    scaler = torch.cuda.amp.GradScaler(
        enabled=use_amp
    )

    for epoch in range(EPOCHS):

        model.train()

        total_loss = 0

        loop = tqdm(train_loader)

        for batch in loop:

            embeddings = batch["embeddings"].to(device)

            attention_mask = batch["attention_mask"].to(device)

            labels = batch["labels"].to(device)

            optimizer.zero_grad()

            with torch.cuda.amp.autocast(
                enabled=use_amp
            ):

                logits = model(
                    embeddings,
                    attention_mask
                )

                loss = criterion(
                    logits,
                    labels
                )

            scaler.scale(loss).backward()

            torch.nn.utils.clip_grad_norm_(
                trainable_params,
                1.0
            )

            scaler.step(optimizer)

            scaler.update()

            scheduler.step()

            total_loss += loss.item()

            loop.set_description(
                f"Epoch {epoch+1}/{EPOCHS}"
            )

            loop.set_postfix(
                loss=loss.item()
            )

        val_result = evaluate(
            model,
            val_loader
        )

        print("\n======================")
        print(f"Epoch {epoch+1}")
        print("======================")

        print(
            f"VAL SUBSET ACC : {val_result['subset_accuracy']:.4f}"
        )

        print(
            f"VAL MICRO P/R/F1: "
            f"{val_result['micro_precision']:.4f} / "
            f"{val_result['micro_recall']:.4f} / "
            f"{val_result['micro_f1']:.4f}"
        )

        print(
            f"VAL MACRO P/R/F1: "
            f"{val_result['macro_precision']:.4f} / "
            f"{val_result['macro_recall']:.4f} / "
            f"{val_result['macro_f1']:.4f}"
        )

        print(
            f"VAL WEIGHTED F1: {val_result['weighted_f1']:.4f}"
        )

        print(
            f"VAL HAMMING LOSS: {val_result['hamming_loss']:.4f}"
        )

    return model

In [18]:
def optimize_thresholds(y_true, y_probs):

    best_thresholds = []

    for i in range(NUM_LABELS):

        best_thr = 0.5
        best_f1 = 0

        for thr in np.arange(0.1, 0.9, 0.05):

            preds = (y_probs[:, i] >= thr).astype(int)

            score = f1_score(
                y_true[:, i],
                preds,
                zero_division=0
            )

            if score > best_f1:

                best_f1 = score

                best_thr = thr

        best_thresholds.append(best_thr)

    return np.array(best_thresholds)

In [19]:
def objective(trial):

    depth = trial.suggest_int(
        "depth",
        1,
        2
    )

    heads = trial.suggest_categorical(
        "heads",
        [2, 4]
    )

    attn_dropout = trial.suggest_float(
        "attn_dropout",
        0.2,
        0.5
    )

    ff_dropout = trial.suggest_float(
        "ff_dropout",
        0.2,
        0.5
    )

    ff_mult = trial.suggest_int(
        "ff_mult",
        2,
        4
    )

    lr = trial.suggest_float(
        "lr",
        1e-5,
        2e-4,
        log=True
    )

    weight_decay = trial.suggest_float(
        "weight_decay",
        1e-4,
        1e-2,
        log=True
    )

    model = EmotionModel(
        depth=depth,
        heads=heads,
        attn_dropout=attn_dropout,
        ff_dropout=ff_dropout,
        ff_mult=ff_mult
    ).to(device)

    model = train_model(
        model,
        train_loader,
        val_loader,
        lr,
        weight_decay
    )

    val_result = evaluate(
        model,
        val_loader
    )

    best_thresholds = optimize_thresholds(
        val_result["labels"],
        val_result["probs"]
    )

    val_result = evaluate(
        model,
        val_loader,
        best_thresholds
    )

    return val_result["macro_f1"]

In [20]:
study = optuna.create_study(
    direction="maximize"
)

study.optimize(
    objective,
    n_trials=N_TRIALS
)


[I 2026-07-16 07:37:45,542] A new study created in memory with name: no-name-211ec1b9-b01a-435d-8970-d60cfc26acb0
Epoch 1/1: 100%|██████████| 850/850 [00:30<00:00, 27.90it/s, loss=0.713]



Epoch 1
VAL SUBSET ACC : 0.1448
VAL MICRO P/R/F1: 0.3981 / 0.7656 / 0.5239
VAL MACRO P/R/F1: 0.3649 / 0.8114 / 0.4782
VAL WEIGHTED F1: 0.5773
VAL HAMMING LOSS: 0.1889


[I 2026-07-16 07:38:28,536] Trial 0 finished with value: 0.5489409755176283 and parameters: {'depth': 1, 'heads': 4, 'attn_dropout': 0.34348691310690604, 'ff_dropout': 0.3606501592118469, 'ff_mult': 4, 'lr': 6.67614445754582e-05, 'weight_decay': 0.0012751900844384712}. Best is trial 0 with value: 0.5489409755176283.
Epoch 1/1: 100%|██████████| 850/850 [00:24<00:00, 35.01it/s, loss=1.03] 



Epoch 1
VAL SUBSET ACC : 0.0442
VAL MICRO P/R/F1: 0.3011 / 0.7050 / 0.4220
VAL MACRO P/R/F1: 0.2669 / 0.7349 / 0.3691
VAL WEIGHTED F1: 0.4849
VAL HAMMING LOSS: 0.2622


[I 2026-07-16 07:39:01,336] Trial 1 finished with value: 0.43372233701958224 and parameters: {'depth': 1, 'heads': 4, 'attn_dropout': 0.489747430421953, 'ff_dropout': 0.22199391756309594, 'ff_mult': 2, 'lr': 1.8816937434353542e-05, 'weight_decay': 0.0019538430882839147}. Best is trial 0 with value: 0.5489409755176283.
Epoch 1/1: 100%|██████████| 850/850 [00:30<00:00, 27.91it/s, loss=0.612]


...

[I 2026-07-16 07:44:10,898] Trial 7 finished with value: 0.43774846930824934 and parameters: {'depth': 2, 'heads': 4, 'attn_dropout': 0.2130247168698701, 'ff_dropout': 0.21775419641070837, 'ff_mult': 3, 'lr': 1.1127630642300795e-05, 'weight_decay': 0.0003242426529428247}. Best is trial 5 with value: 0.6318188460865397.
Epoch 1/1: 100%|██████████| 850/850 [00:46<00:00, 18.38it/s, loss=0.936]



Epoch 1
VAL SUBSET ACC : 0.1477
VAL MICRO P/R/F1: 0.3980 / 0.7634 / 0.5232
VAL MACRO P/R/F1: 0.3587 / 0.8023 / 0.4728
VAL WEIGHTED F1: 0.5744
VAL HAMMING LOSS: 0.1889


[I 2026-07-16 07:45:15,920] Trial 8 finished with value: 0.5488188434086793 and parameters: {'depth': 2, 'heads': 4, 'attn_dropout': 0.3981306788784207, 'ff_dropout': 0.31601424764863545, 'ff_mult': 3, 'lr': 4.1192704360615184e-05, 'weight_decay': 0.00017830387807844854}. Best is trial 5 with value: 0.6318188460865397.
Epoch 1/1: 100%|██████████| 850/850 [00:52<00:00, 16.29it/s, loss=0.81] 



Epoch 1
VAL SUBSET ACC : 0.1533
VAL MICRO P/R/F1: 0.4050 / 0.7669 / 0.5301
VAL MACRO P/R/F1: 0.3683 / 0.8039 / 0.4820
VAL WEIGHTED F1: 0.5812
VAL HAMMING LOSS: 0.1846


[I 2026-07-16 07:46:30,552] Trial 9 finished with value: 0.5514677377273713 and parameters: {'depth': 2, 'heads': 4, 'attn_dropout': 0.21984914606160977, 'ff_dropout': 0.20545353053556487, 'ff_mult': 4, 'lr': 4.321512489820533e-05, 'weight_decay': 0.00040911669511505347}. Best is trial 5 with value: 0.6318188460865397.


In [21]:
print("\n======================")
print("BEST PARAMETERS")
print("======================")

print(study.best_params)

best_params = study.best_params


BEST PARAMETERS
{'depth': 2, 'heads': 4, 'attn_dropout': 0.2691450563663464, 'ff_dropout': 0.28475714690185233, 'ff_mult': 4, 'lr': 0.0001955577124628116, 'weight_decay': 0.0015755090667191358}


In [22]:
final_model = EmotionModel(
    depth=best_params["depth"],
    heads=best_params["heads"],
    attn_dropout=best_params["attn_dropout"],
    ff_dropout=best_params["ff_dropout"],
    ff_mult=best_params["ff_mult"]
).to(device)

In [23]:
final_model = train_model(
    final_model,
    train_loader,
    val_loader,
    best_params["lr"],
    best_params["weight_decay"]
)

Epoch 1/1: 100%|██████████| 850/850 [00:52<00:00, 16.28it/s, loss=0.554]



Epoch 1
VAL SUBSET ACC : 0.2391
VAL MICRO P/R/F1: 0.4738 / 0.8029 / 0.5959
VAL MACRO P/R/F1: 0.4265 / 0.8432 / 0.5460
VAL WEIGHTED F1: 0.6360
VAL HAMMING LOSS: 0.1478


In [24]:
val_result = evaluate(
    final_model,
    val_loader
)

best_thresholds = optimize_thresholds(
    val_result["labels"],
    val_result["probs"]
)

print("\n======================")
print("BEST THRESHOLDS")
print("======================")

for label, thr in zip(
    LABEL_COLUMNS,
    best_thresholds
):
    print(f"{label}: {thr:.2f}")


BEST THRESHOLDS
politics: 0.35
human_rights: 0.60
quality_of_life: 0.65
international: 0.70
social: 0.70
environment: 0.80
economics: 0.80
culture: 0.80
labor: 0.80
national_security: 0.80
ict: 0.85
education: 0.85


In [25]:
test_result = evaluate(
    final_model,
    test_loader,
    best_thresholds
)

print("\n======================")
print("FINAL RESULT ON OFFICIAL PRACHATAI TEST SET")
print("======================")

print(
    f"Subset Accuracy / Exact Match : {test_result['subset_accuracy']:.4f}"
)

print(
    f"Micro Precision              : {test_result['micro_precision']:.4f}"
)

print(
    f"Micro Recall                 : {test_result['micro_recall']:.4f}"
)

print(
    f"Micro F1                     : {test_result['micro_f1']:.4f}"
)

print(
    f"Macro Precision              : {test_result['macro_precision']:.4f}"
)

print(
    f"Macro Recall                 : {test_result['macro_recall']:.4f}"
)

print(
    f"Macro F1                     : {test_result['macro_f1']:.4f}"
)

print(
    f"Weighted F1                  : {test_result['weighted_f1']:.4f}"
)

print(
    f"Hamming Loss                 : {test_result['hamming_loss']:.4f}"
)



FINAL RESULT ON OFFICIAL PRACHATAI TEST SET
Subset Accuracy / Exact Match : 0.3517
Micro Precision              : 0.6023
Micro Recall                 : 0.7563
Micro F1                     : 0.6706
Macro Precision              : 0.5471
Macro Recall                 : 0.7100
Macro F1                     : 0.6165
Weighted F1                  : 0.6762
Hamming Loss                 : 0.1006


In [26]:
print("\n======================")
print("CLASSIFICATION REPORT")
print("======================")

print(
    classification_report(
        test_result["labels"],
        test_result["preds"],
        target_names=LABEL_COLUMNS,
        zero_division=0
    )
)


CLASSIFICATION REPORT
                   precision    recall  f1-score   support

         politics       0.74      0.86      0.80      3842
     human_rights       0.59      0.68      0.63      1511
  quality_of_life       0.56      0.75      0.64      1127
    international       0.63      0.84      0.72       834
           social       0.33      0.51      0.41       789
      environment       0.65      0.74      0.69       772
        economics       0.51      0.66      0.57       519
          culture       0.42      0.61      0.50       398
            labor       0.73      0.88      0.80       350
national_security       0.41      0.63      0.49       338
              ict       0.56      0.78      0.65       292
        education       0.42      0.57      0.49       255

        micro avg       0.60      0.76      0.67     11027
        macro avg       0.55      0.71      0.62     11027
     weighted avg       0.61      0.76      0.68     11027
      samples avg       0.65   

In [27]:
SAVE_PATH = "/kaggle/working/final_prachatai_wangchanberta_xtransformers.pt"

torch.save(
    {
        "model_state_dict": final_model.state_dict(),
        "thresholds": best_thresholds,
        "best_params": best_params,
        "labels": LABEL_COLUMNS,
        "model_name": MODEL_NAME,
        "max_len": MAX_LEN,
        "note": "Prachatai official Train/Validation/Test; cached frozen-encoder embeddings; 100 epochs."
    },
    SAVE_PATH
)

print("\nMODEL SAVED:", SAVE_PATH)


MODEL SAVED: /kaggle/working/final_prachatai_wangchanberta_xtransformers.pt
